Predicting baby name sex using PyTorch for Logistic Regression

### References

- [Logistic Regression with PyTorch](https://proai.org/pytorch-logistic-regression) by [Denny Loevlie](https://twitter.com/DennisLoevlie)
- [PyTorch OO design compared with SkLearn](https://jeancochrane.com/blog/pytorch-functional-api) by [Jean Cochrane](https://JeanCochrane.com)
- [Building Your First Network in PyTorch](https://t.co/m9I4e0tfrC) by [Ta-Ying Cheng](https://www.linkedin.com/in/tim-ta-ying-cheng-411857139/)
- [data.world US baby names since 1880](https://data.world/ssa/baby-names-for-us-states-territories)



In [245]:
import numpy as np
import pandas as pd
# neither year nor len are statistically significant predictors of sex
from pathlib import Path

In [246]:
pd.options.display.max_rows=7

In [247]:
CWD = Path('.').absolute().resolve()
DATA_DIR_NAME = '.nlpia2-data'
DATA_FILE = 'baby-names-region.csv.gz'
CWD

PosixPath('/home/hobs/code/tangibleai/nlpia2/src/nlpia2/ch05')

In [248]:
parent = CWD
data_dir = parent / DATA_DIR_NAME 
filepath = data_dir / DATA_FILE
for i in range(10):
    print(filepath)
    if filepath.is_file():
        break
    parent = parent.parent
    data_dir = parent / DATA_DIR_NAME 
    filepath = data_dir / DATA_FILE
filepath

/home/hobs/code/tangibleai/nlpia2/src/nlpia2/ch05/.nlpia2-data/baby-names-region.csv.gz
/home/hobs/code/tangibleai/nlpia2/src/nlpia2/.nlpia2-data/baby-names-region.csv.gz
/home/hobs/code/tangibleai/nlpia2/src/.nlpia2-data/baby-names-region.csv.gz
/home/hobs/code/tangibleai/nlpia2/.nlpia2-data/baby-names-region.csv.gz


PosixPath('/home/hobs/code/tangibleai/nlpia2/.nlpia2-data/baby-names-region.csv.gz')

In [249]:
def find_dir(dirname=DATA_DIR_NAME, parent=Path('.').absolute().resolve(), max_parents=20):
    for i in range(max_parents):
        data_dir = parent / dirname
        if data_dir.is_dir():
            return data_dir
        parent = parent.parent

DATA_DIR = find_dir()
DATA_DIR

PosixPath('/home/hobs/code/tangibleai/nlpia2/.nlpia2-data')

In [250]:
def find_file(filename, parent=Path('.').absolute().resolve(), data_dir_name=DATA_DIR_NAME, max_parents=20):
    for i in range(max_parents):
        data_dir = parent / data_dir_name 
        filepath = data_dir / filename
        if filepath.is_file():
            return filepath
        parent = parent.parent

filepath = find_file(DATA_FILE)
filepath

PosixPath('/home/hobs/code/tangibleai/nlpia2/.nlpia2-data/baby-names-region.csv.gz')

In [251]:
df = pd.read_csv(filepath)

In [252]:
np.random.seed(451)
df = df.sample(10_000)
df

,region,sex,year,name,count,freq
6139665,WV,F,1987,Brittani,10,0.000003
2565339,MD,F,1954,Ida,18,0.000005
22297,AK,M,1988,Maxwell,5,0.000001
...,...,...,...,...,...,...
4475894,OK,F,1950,Leah,9,0.000003
5744351,VA,F,2007,Carley,11,0.000003
5583882,TX,M,2019,Kartier,10,0.000003


In [253]:
names = df['name'].unique()
list(names[:10])

['Brittani',
 'Ida',
 'Maxwell',
 'Charlene',
 'Todd',
 'Aubrey',
 'Arianna',
 'Otis',
 'Trenton',
 'Faustino']

In [254]:
len(names) / len(df)

0.4025

In [255]:
# df = pd.get_dummies(df, columns=['region'])
# df.head()

In [256]:
df = df.rename(dict(name='name_', sex='sex_'), axis=1)
df.head()

,region,sex_,year,name_,count,freq
6139665,WV,F,1987,Brittani,10,0.000003
2565339,MD,F,1954,Ida,18,0.000005
22297,AK,M,1988,Maxwell,5,0.000001
5114650,TN,F,1972,Charlene,24,0.000008
2126395,KS,M,1954,Todd,11,0.000003


In [257]:
df = df.groupby(['name_', 'sex_']).sum()
df.head()

,,year,count,freq
name_,sex_,,,
Aaden,M,2008,51,0.000015
Aahana,F,2018,26,0.000009
Aahil,M,2019,5,0.000002
Aaleyah,F,2010,17,0.000005
Aalia,F,4033,13,0.000004


In [258]:
df['name'] = df.index.get_level_values('name_')
df['sex'] = df.index.get_level_values('sex_')
df.head()

,,year,count,freq,name,sex
name_,sex_,,,,,
Aaden,M,2008,51,0.000015,Aaden,M
Aahana,F,2018,26,0.000009,Aahana,F
Aahil,M,2019,5,0.000002,Aahil,M
Aaleyah,F,2010,17,0.000005,Aaleyah,F
Aalia,F,4033,13,0.000004,Aalia,F


In [259]:
df.query('name == "Chris"')

year  count      freq   name sex
name_ sex_                                  
Chris F     1983      5  0.000002  Chris   F
      M     7850    239  0.000069  Chris   M

In [260]:
df.loc[pd.IndexSlice['Chris', :]]

,year,count,freq,name,sex
sex_,,,,,
F,1983,5,0.000002,Chris,F
M,7850,239,0.000069,Chris,M


In [261]:
df['istrain'] = np.random.rand(len(df)) < .9
df.head()

,,year,count,freq,name,sex,istrain
name_,sex_,,,,,,
Aaden,M,2008,51,0.000015,Aaden,M,True
Aahana,F,2018,26,0.000009,Aahana,F,True
Aahil,M,2019,5,0.000002,Aahil,M,True
Aaleyah,F,2010,17,0.000005,Aaleyah,F,True
Aalia,F,4033,13,0.000004,Aalia,F,True


In [262]:
# A list of dicts or a dict of dicts is fastest way to create dataframe from groups of rows
# https://stackoverflow.com/users/8727339/mohit-motwani
# https://stackoverflow.com/a/57001947/623735

df_most_common = {}
for name, group in df.groupby('name'):
    row_dict = group.iloc[group['count'].argmax()].to_dict()
    df_most_common[(name, row_dict['sex'])] = row_dict
df_most_common = pd.DataFrame(df_most_common).T
df_most_common

,,year,count,freq,name,sex,istrain
Aaden,M,2008,51,0.000015,Aaden,M,True
Aahana,F,2018,26,0.000009,Aahana,F,True
Aahil,M,2019,5,0.000002,Aahil,M,True
...,...,...,...,...,...,...,...
Zvi,M,2015,5,0.000002,Zvi,M,True
Zya,F,2019,8,0.000003,Zya,F,True
Zylah,F,2008,5,0.000001,Zylah,F,True


In [263]:
df_most_common['istest'] = ~df_most_common['istrain'].astype(bool)
df_most_common.head()

,,year,count,freq,name,sex,istrain,istest
Aaden,M,2008,51,0.000015,Aaden,M,True,False
Aahana,F,2018,26,0.000009,Aahana,F,True,False
Aahil,M,2019,5,0.000002,Aahil,M,True,False
Aaleyah,F,2010,17,0.000005,Aaleyah,F,True,False
Aalia,F,4033,13,0.000004,Aalia,F,True,False


In [264]:
df_most_common[['istest', 'istrain']].sum() / len(df_most_common)

istest     0.095652
istrain    0.904348
dtype: object

In [265]:
istest = df_most_common['istest']
istest

Aaden   M    False
Aahana  F    False
Aahil   M    False
             ...  
Zvi     M    False
Zya     F    False
Zylah   F    False
Name: istest, Length: 4025, dtype: bool

In [266]:
istest.sum()

385

In [267]:
istest_idx = df_most_common[istest].index
istest_idx[:4]

MultiIndex([('Abelardo', 'M'),
            ( 'Adaline', 'F'),
            ( 'Adalynn', 'F'),
            ( 'Addelyn', 'F')],
           )

In [268]:
df['istrain'].sum() / len(df)

0.9042000943841435

In [269]:
df['istest'] = df_most_common['istest']
df['istest'] = df['istest'].fillna(False)
df['istrain'] = ~df['istest']
df['istrain'].sum() / len(df)

0.9091552619159982

In [270]:
istrain = df['istrain']
del df['istrain']
del df['istest']
istrain.sum() / len(istrain)

0.9091552619159982

In [271]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(1, 3), lowercase=False)
vectorizer

TfidfVectorizer(analyzer='char', lowercase=False, ngram_range=(1, 3))

In [272]:
vectorizer.fit(df['name'][istrain])

TfidfVectorizer(analyzer='char', lowercase=False, ngram_range=(1, 3))

In [273]:
vecs = vectorizer.transform(df['name'])
vecs = pd.DataFrame.sparse.from_spmatrix(vecs)
vecs.head()


,0,1,2,3,4,5,6,7,8,9,...,3653,3654,3655,3656,3657,3658,3659,3660,3661,3662
0,0.194050,0.396454,0.507346,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.182739,0.373346,0.000000,0.455153,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.185394,0.378769,0.000000,0.461765,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.166819,0.340819,0.000000,0.000000,0.389483,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.214608,0.438454,0.000000,0.000000,0.501059,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [274]:
vecs.columns = vectorizer.get_feature_names_out()
vecs.index = df.index
vecs.head()[vecs.columns[:5]]

,,A,Aa,Aad,Aah,Aal
name_,sex_,,,,,
Aaden,M,0.194050,0.396454,0.507346,0.000000,0.000000
Aahana,F,0.182739,0.373346,0.000000,0.455153,0.000000
Aahil,M,0.185394,0.378769,0.000000,0.461765,0.000000
Aaleyah,F,0.166819,0.340819,0.000000,0.000000,0.389483
Aalia,F,0.214608,0.438454,0.000000,0.000000,0.501059


In [275]:
vecs.shape

(4238, 3663)

In [276]:
import torch
torch

<module 'torch' from '/home/hobs/anaconda3/envs/nlpia2/lib/python3.8/site-packages/torch/__init__.py'>

In [277]:
class LogisticRegressionNN(torch.nn.Module):

    def __init__(self, num_features, num_outputs=1):
         super().__init__()
         self.linear = torch.nn.Linear(num_features, num_outputs)

    def forward(self, X):
        return torch.sigmoid(self.linear(X))

In [278]:
def make_tensor(X):
    """ Convert numpy ndarray to torch.Tensor """
    X = getattr(X, 'values', X)
    return X if isinstance(X, torch.Tensor) else torch.Tensor(X)

def make_array(x):
    """ Convert torch.Tensor to numpy 1-D array """
    if hasattr(x, 'detach'):
        return torch.squeeze(x).detach().numpy()
    return x

In [279]:
num_features = vecs.shape[1]  # number of unique n-grams in our "vocabulary"
num_outputs = 1    # number of nesses (sexes) to predict, we're predicting only femaleness

In [280]:
from tqdm import tqdm
import time
import json
import copy

# Fraction of the tensors y_pred and y that are the same 
# (y_pred == y).sum() / len(y)
def measure_binary_accuracy(y_pred, y):
    """ Round y_pred and y then count the preds that are equal to the truth to compute fraction correct """
    y_pred = make_array(y_pred).round()
    y = make_array(y).round()
    num_correct = (y_pred == y).sum()
    return num_correct / len(y)

In [281]:
def measure_performance(model, X_train, X_test, y_train, y_test, criterion):
    with torch.no_grad():
        # Calculating the loss and accuracy for the train dataset
        accuracy_train = measure_binary_accuracy(model(X_train), y_train)
        outputs_test = torch.squeeze(model(X_test))
        accuracy_test = measure_binary_accuracy(outputs_test, y_test)
        loss_test = criterion(outputs_test, y_test)
        return dict(i=i, 
                    # loss_train=loss.item(),
                    accuracy_train=accuracy_train,
                    loss_test=loss_test.item(),
                    accuracy_test=accuracy_test)

In [282]:
model = LogisticRegressionNN(num_features=vecs.shape[1], num_outputs=1)
model

LogisticRegressionNN(
  (linear): Linear(in_features=3663, out_features=1, bias=True)
)

In [283]:
learning_rate = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
optimizer

SGD (
Parameter Group 0
    dampening: 0
    lr: 0.01
    momentum: 0
    nesterov: False
    weight_decay: 0
)

In [284]:
# BCE: Binary Cross Entropy
criterion = torch.nn.BCELoss(weight=torch.Tensor(df[['count']].values))
criterion

BCELoss()

In [285]:
df['majority_sex'] = df['sex']
for name_, sex_ in df_most_common.index:
    opposite_sex = 'F' if sex_ == 'M' else 'M'
    try:
        df.loc[(name_, opposite_sex)]['majority_sex'] = sex_
    except KeyError:
        pass
y_pred = df[['majority_sex']] == 'F'
y = df[['sex']] == 'F'
(y.values == y_pred.values).sum() / len(y)

/home/hobs/anaconda3/envs/nlpia2/lib/python3.8/site-packages/pandas/core/series.py:1056: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cacher_needs_updating = self._check_is_chained_assignment_possible()


1.0

In [ ]:
weight_train = torch.Tensor(df[['count']].loc[df_most_common.index].values)
criterion = torch.nn.BCELoss(weight=weight_test)
criterion

In [ ]:
def rand_range(min_value=0.001, max_value=1):
    scale = max_value - min_value
    return scale * np.random.rand() + min_value

In [ ]:
def rand_range_log(min_value=0.001, max_value=1):
    min_log = np.log(min_value)
    max_log = np.log(max_value)
    return np.exp(rand_range(np.log(min_value), np.log(max_value)))

Create random hyperparameter table for optimizer learning_rate and momentum

In [ ]:
# lr: learning_rate
hyperparam_ranges = dict(lr=[0.001, 1.0], momentum=[0.00001, 1.0])
hyperparam_table = []
num_attempts = 30
for i in range(num_attempts):
    hyperparam_values = dict()
    for k, v in hyperparam_ranges.items():
        hyperparam_values[k] = rand_range_log(*hyperparam_ranges[k])
    hyperparam_table.append(hyperparam_values)
pd.DataFrame(hyperparam_table)

In [ ]:
model = LogisticRegressionNN(num_features=vecs.shape[1], num_outputs=1)
model

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), **hyperparam_table[0])
optimizer

In [ ]:
weight = torch.Tensor(df['count'][istrain].values.flatten())
weight

In [ ]:
# pbar = tqdm(hyperparam_table, desc='Training attempt', total=len(hyperparam_table))
num_epochs=2000

t0 = time.time()
for i, hyperparam_values in enumerate(hyperparam_table):
    t1 = time.time()
    model = LogisticRegressionNN(num_features=vecs.shape[1], num_outputs=1)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    # BCE: Binary Cross Entropy weighted by the number of babies with that first name and sex
    criterion_train = torch.nn.BCELoss(weight=torch.Tensor(df[['count']][istrain].values))
    criterion_test = torch.nn.BCELoss(weight=torch.Tensor(df[['count']][~istrain].values))
    X = vecs.values
    y = (df[['sex']] == 'F').values
    X_train = torch.Tensor(X[istrain])
    X_test = torch.Tensor(X[~istrain])
    y_train = torch.Tensor(y[istrain])
    y_test = torch.Tensor(y[~istrain])

    pbar_epochs = tqdm(range(num_epochs), desc='Epoch:', total=num_epochs)
    results = [None] * num_epochs
    for epoch in pbar_epochs:
        optimizer.zero_grad() # Setting our stored gradients equal to zero
        outputs = model(X_train)
        loss_train = criterion_train(outputs, y_train) 
        loss_train.backward() # Computes the gradient of the given tensor w.r.t. the weights/bias
        loss_train = loss_train.item()
        optimizer.step() # Updates weights and biases with the optimizer (SGD)
        # print(f'Train loss: {np.round(loss_train.detach().numpy(), 4):0.4f}')
        outputs_test = model(X_test)
        loss_test = criterion_test(outputs_test, y_test).item()
        accuracy_test = measure_binary_accuracy(outputs_test, y_test)
        results[epoch] = dict(loss_train=loss_train, loss_test=loss_test, accuracy_test=accuracy_test)
        # pbar_epochs.set_description(f'loss_train/test: {loss_train:.4f}/{loss_test:.4f}')
    t2 = time.time()
    results[-1]['time_per_attempt'] = t2 - t1
    results[-1]['total_time'] = t2 - t0
    hyperparam_table[i].update(results[-1])
    print(f'attempt {i:04d}/{len(hyperparam_table)}')
    for k, v in hyperparam_table[i]:
        print(f'{k}: {v:04d}')

In [ ]:
hyperparam_table[-1]['loss_train'].item()



In [ ]:
hyperparam_df = pd.DataFrame(hyperparam_table)
hyperparam_df

In [ ]:
def fit(model=model, X=vecs.values, y=(df[['sex']] == 'F').values, optimizer=None,
        num_epochs=30, learning_rate=.1, criterion=criterion, optimizer=optimizer):
    pbar = tqdm(range(num_epochs), desc='Epoch', total=num_epochs)
    X_train = torch.Tensor(X[istrain])
    X_test = torch.Tensor(X[~istrain])
    y_train = torch.Tensor(y[istrain])
    y_test = torch.Tensor(y[~istrain])

    results = []
    for i in pbar:
        optimizer.zero_grad() # Setting our stored gradients equal to zero
        outputs = model(X_train)
        loss_train = criterion(outputs, y_train) 
        loss_train.backward() # Computes the gradient of the given tensor w.r.t. the weights/bias
        optimizer.step() # Updates weights and biases with the optimizer (SGD)
    return results

In [ ]:
results = fit()

In [ ]:
pd.DataFrame(results)

In [ ]:
# model.score(vecs[~istrain], y[~istrain], sample_weight=df['count'][~istrain])

In [ ]:
# model.classes_


In [ ]:
names = ['Dewey', 'Kemal', 'Copeland', 'Vishvesh']
ourvecs = vectorizer.transform(names)
ourvecs = pd.DataFrame.sparse.from_spmatrix(ourvecs)
ourvecs.columns = vectorizer.get_feature_names_out()
ourvecs.index = list(zip(names, 'M'*len(names)))
ourvecs

In [ ]:
ourtensors = 

In [ ]:
names = ['Maria', 'Syndee', 'Aditi', 'Constance']
vecs = vectorizer.transform(names)
vecs = pd.DataFrame.sparse.from_spmatrix(vecs)
vecs.columns = vectorizer.get_feature_names_out()
vecs.index = list(zip(names, 'M'*len(names)))
pd.DataFrame(model.predict_proba(vecs)[:,0], index=vecs.index)

In [ ]:
class LogisticRegressionNumpyNN(LogisticRegressionNN):

    def __init__(self, *args, **kwargs):
         super().__init__(*args, **kwargs)

    def predict_proba(self, X):
        return self.forward(make_tensor(X))
    
    def predict(self, X):
        return (np.array(self.forward(make_tesnor(X))) > 0.5).astype(int)
    
# ', '.join([v for v in dir(LogisticRegression) if v[0] != '_'])